# Hemo Invasion PDE Demo

This tutorial shows how to run `HemoInvasion3D` in TumorTwin using the same pattern as `HGG_Demo` and `TNBC_Demo`.

We include a **mini 50-day run** for quick sanity checking.

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from tumortwin.models import HemoInvasion3D, extract_trajectory_component
from tumortwin.optimizers import LMoptimizer, LMoptions
from tumortwin.postprocessing import (
    compute_total_cell_count,
    plot_cellularity_map,
    plot_imaging_summary,
    plot_patient_timeline,
    plot_predicted_TCC,
)
from tumortwin.preprocessing import ADC_to_cellularity, compute_carrying_capacity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import CropSettings, CropTarget
from tumortwin.types.hgg_data import HGGPatientData
from tumortwin.utils import days_since_first

In [ ]:
# Choose device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Resolve paths robustly (works whether cwd is repo root or tutorials/)
cwd = Path.cwd()
repo_root = cwd if (cwd / "tumortwin").exists() else cwd.parent

patient_json = repo_root / "input_files" / "HGG_demo_001" / "HGG_demo_001.json"
if not patient_json.exists():
    raise FileNotFoundError(f"Could not find patient json: {patient_json}")

crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)
patient_data = HGGPatientData.from_file(patient_json, crop_settings=crop_settings)
carrying_capacity = compute_carrying_capacity(patient_data.brainmask_image)

print(f"Patient: {patient_data.patient}")
print(f"Visits: {len(patient_data.visits)}")
print(f"Grid shape: {patient_data.brainmask_image.array.shape}")
print(f"Carrying capacity: {carrying_capacity}")

In [ ]:
# Build initial fields from first visit
visit0 = patient_data.visits[0]

cellularity0 = ADC_to_cellularity(
    visit0.adc_image,
    visit0.roi_enhance_image,
    visit0.roi_nonenhance_image,
)

initial_n = torch.from_numpy(cellularity0.array).float().to(device)
initial_m = torch.zeros_like(initial_n)
initial_s = torch.ones_like(initial_n)

# We use ROI-enhancing voxels as a proxy vessel mask for the demo.
# In a production setup, replace this with a proper vessel segmentation/mask.
vessel_mask = torch.from_numpy((visit0.roi_enhance_image.array > 0).astype(np.bool_)).to(device)

print("Initial tensors:")
print("  n:", tuple(initial_n.shape), f"[{initial_n.min().item():.3f}, {initial_n.max().item():.3f}]")
print("  m:", tuple(initial_m.shape), f"[{initial_m.min().item():.3f}, {initial_m.max().item():.3f}]")
print("  s:", tuple(initial_s.shape), f"[{initial_s.min().item():.3f}, {initial_s.max().item():.3f}]")
print("  vessel voxels:", int(vessel_mask.sum().item()))

In [ ]:
# Initialize HemoInvasion3D model with extra-conservative defaults
model = HemoInvasion3D(
    B=torch.tensor(0.010, dtype=torch.float32, device=device),
    Dn=torch.tensor(0.001, dtype=torch.float32, device=device),
    Ds=torch.tensor(0.015, dtype=torch.float32, device=device),
    k_s=torch.tensor(0.040, dtype=torch.float32, device=device),
    s_star=torch.tensor(0.250, dtype=torch.float32, device=device),
    patient_data=patient_data,
    initial_n=initial_n,
    initial_m=initial_m,
    initial_s=initial_s,
    K=torch.tensor(1.0, dtype=torch.float32, device=device),
    s_crit=torch.tensor(0.35, dtype=torch.float32, device=device),
    s_smooth=torch.tensor(0.08, dtype=torch.float32, device=device),
    s_outside=0.0,
    s_vessel=1.0,
    vessel_mask=vessel_mask,
    time_scale_days=120.0,
    poisson_iterations=32,
    require_grad=False,
    device=device,
)

solver = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=0.02),
        method="rk4",
        device=device,
        use_adjoint=False,
    ),
)

u0 = model.get_initial_state()
print("u0 shape:", tuple(u0.shape))

In [ ]:
# Mini run: 50 days (daily outputs)
mini_t0 = patient_data.visits[0].time
mini_timepoints = [mini_t0 + timedelta(days=d) for d in range(0, 51)]  # 0..50 days

times_mini, traj_mini = solver.solve(timepoints=mini_timepoints, u_initial=u0)

# Unpack fields: traj shape = (T, 3, D, H, W)
n_series = extract_trajectory_component(traj_mini, 0)
m_series = extract_trajectory_component(traj_mini, 1)
s_series = extract_trajectory_component(traj_mini, 2)

# Physical projection for diagnostics/plots (state constraints)
n_series_phys = torch.clamp(n_series, 0.0, 1.0)
m_series_phys = torch.clamp(m_series, 0.0, 1.0)
s_series_phys = torch.clamp(s_series, 0.0, 1.0)

print("Mini run complete")
print("  trajectory shape:", tuple(traj_mini.shape))
print("  n finite:", bool(torch.isfinite(n_series).all()))
print("  m finite:", bool(torch.isfinite(m_series).all()))
print("  s finite:", bool(torch.isfinite(s_series).all()))

In [ ]:
# Fast sanity checks on dynamics
mass_n = n_series_phys.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mass_m = m_series_phys.sum(dim=(1, 2, 3)).detach().cpu().numpy()
mean_s = s_series_phys.mean(dim=(1, 2, 3)).detach().cpu().numpy()
time_days = times_mini.detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(time_days, mass_n)
axes[0].set_title("Total proliferating cells (n)")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_days, mass_m)
axes[1].set_title("Total quiescent cells (m)")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_days, mean_s)
axes[2].set_title("Mean substrate (S)")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

raw_n_min, raw_n_max = n_series.min().item(), n_series.max().item()
raw_m_min, raw_m_max = m_series.min().item(), m_series.max().item()
raw_s_min, raw_s_max = s_series.min().item(), s_series.max().item()

print("Raw ranges (before projection):")
print(f"  n: [{raw_n_min:.4f}, {raw_n_max:.4f}]")
print(f"  m: [{raw_m_min:.4f}, {raw_m_max:.4f}]")
print(f"  S: [{raw_s_min:.4f}, {raw_s_max:.4f}]")
print("Projected ranges (physical):")
print(f"  n: [{n_series_phys.min().item():.4f}, {n_series_phys.max().item():.4f}]")
print(f"  m: [{m_series_phys.min().item():.4f}, {m_series_phys.max().item():.4f}]")
print(f"  S: [{s_series_phys.min().item():.4f}, {s_series_phys.max().item():.4f}]")

# Stability warning for the raw trajectory
max_abs_raw = max(abs(raw_n_min), abs(raw_n_max), abs(raw_m_min), abs(raw_m_max))
if max_abs_raw > 5.0:
    print("WARNING: raw trajectory appears unstable (|n| or |m| > 5).")
    print("Try smaller solver step_size and/or more conservative parameters.")

In [ ]:
# Visual check: center slice for n, m, S at day 0 and day 50
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _show(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_show(axes[0, 0], n_series_phys[0, z].detach().cpu().numpy(), "n (day 0)", "magma")
_show(axes[0, 1], m_series_phys[0, z].detach().cpu().numpy(), "m (day 0)", "viridis")
_show(axes[0, 2], s_series_phys[0, z].detach().cpu().numpy(), "S (day 0)", "plasma")

_show(axes[1, 0], n_series_phys[-1, z].detach().cpu().numpy(), "n (day 50)", "magma")
_show(axes[1, 1], m_series_phys[-1, z].detach().cpu().numpy(), "m (day 50)", "viridis")
_show(axes[1, 2], s_series_phys[-1, z].detach().cpu().numpy(), "S (day 50)", "plasma")

plt.tight_layout()

## Notes

- This mini run is intended for **fast model sanity checks** only.
- For calibration/forecasting, tune parameters (`B`, `Dn`, `Ds`, `k_s`, transition parameters) and use patient-specific vascular masks if available.
- You can increase horizon and reduce `step_size` after the short run looks stable.

## Full run to last visit

This section mirrors `HGG_Demo` / `TNBC_Demo`: integrate from first to last visit with denser output.  
Then we compare the first 50 days of this full run against the mini-run.

In [ ]:
# Full run: first visit -> last visit (0.5-day output)
full_t0 = patient_data.visits[0].time
full_t1 = patient_data.visits[-1].time

full_timepoints = []
cur_t = full_t0
while cur_t <= full_t1:
    full_timepoints.append(cur_t)
    cur_t += timedelta(days=0.5)
if full_timepoints[-1] != full_t1:
    full_timepoints.append(full_t1)

times_full, traj_full = solver.solve(timepoints=full_timepoints, u_initial=u0)

n_full = extract_trajectory_component(traj_full, 0)
m_full = extract_trajectory_component(traj_full, 1)
s_full = extract_trajectory_component(traj_full, 2)

print("Full run complete")
print("  visits window (days):", (full_t1 - full_t0).days)
print("  output points:", len(full_timepoints))
print("  trajectory shape:", tuple(traj_full.shape))
print("  all finite:", bool(torch.isfinite(traj_full).all()))

In [ ]:
# Compare mini-run vs first 50 days of full-run
full_days = times_full.detach().cpu().numpy()
mini_days = times_mini.detach().cpu().numpy()

mask_50 = full_days <= 50.0
n_mass_full_50 = n_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_full_50 = m_full[mask_50].sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_full_50 = s_full[mask_50].mean(dim=(1, 2, 3)).detach().cpu().numpy()

n_mass_mini = n_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
m_mass_mini = m_series.sum(dim=(1, 2, 3)).detach().cpu().numpy()
s_mean_mini = s_series.mean(dim=(1, 2, 3)).detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(full_days[mask_50], n_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[0].plot(mini_days, n_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[0].set_title("Total n")
axes[0].set_xlabel("days")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(full_days[mask_50], m_mass_full_50, label="full run (<=50d)", alpha=0.9)
axes[1].plot(mini_days, m_mass_mini, "--", label="mini run 50d", alpha=0.9)
axes[1].set_title("Total m")
axes[1].set_xlabel("days")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(full_days[mask_50], s_mean_full_50, label="full run (<=50d)", alpha=0.9)
axes[2].plot(mini_days, s_mean_mini, "--", label="mini run 50d", alpha=0.9)
axes[2].set_title("Mean S")
axes[2].set_xlabel("days")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()

# Visual compare center slice at ~day 50
idx50_full = int(np.argmin(np.abs(full_days - 50.0)))
idx50_mini = int(np.argmin(np.abs(mini_days - 50.0)))
z = n_series.shape[1] // 2

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

def _imshow(ax, arr, title, cmap):
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

_imshow(axes[0, 0], n_full[idx50_full, z].detach().cpu().numpy(), "n full ~day50", "magma")
_imshow(axes[0, 1], m_full[idx50_full, z].detach().cpu().numpy(), "m full ~day50", "viridis")
_imshow(axes[0, 2], s_full[idx50_full, z].detach().cpu().numpy(), "S full ~day50", "plasma")

_imshow(axes[1, 0], n_series[idx50_mini, z].detach().cpu().numpy(), "n mini day50", "magma")
_imshow(axes[1, 1], m_series[idx50_mini, z].detach().cpu().numpy(), "m mini day50", "viridis")
_imshow(axes[1, 2], s_series[idx50_mini, z].detach().cpu().numpy(), "S mini day50", "plasma")

plt.tight_layout()

## Parameter calibration (LM) on early visits

Below is a lightweight calibration workflow inspired by `HGG_Demo` and `TNBC_Demo`:

- use early visits as targets,
- build a better start point with analytic init + coarse grid search,
- optimize a small set of PDE parameters,
- track loss and compare pre/post calibration trajectories.

To keep runtime practical, the default example calibrates only 3 parameters (`B`, `Dn`, `k_s`) on the first 3 visits.

In [ ]:
# Build measured target maps at visit times
measured_cellularity_maps = [
    ADC_to_cellularity(v.adc_image, v.roi_enhance_image, v.roi_nonenhance_image)
    for v in patient_data.visits
]

n_visits_calibration = min(3, len(patient_data.visits))  # include initial visit
target_timepoints = [v.time for v in patient_data.visits[:n_visits_calibration]]

y_target = torch.stack(
    [
        torch.from_numpy(measured_cellularity_maps[i].array).float().to(device)
        for i in range(n_visits_calibration)
    ],
    dim=0,
)


def downsample_field(field, factor=2):
    """Downsample spatial axes by integer factor (supports 3D or T+3D tensors)."""
    if factor <= 1:
        return field
    if field.dim() == 3:
        return field[::factor, ::factor, ::factor]
    if field.dim() == 4:
        return field[:, ::factor, ::factor, ::factor]
    raise ValueError(f"Unsupported field shape {tuple(field.shape)}")


COARSE_FACTOR = 2
y_target_small = downsample_field(y_target, COARSE_FACTOR)

print("Calibration targets prepared")
print("  visits used:", n_visits_calibration)
print("  target shape:", tuple(y_target.shape))
print("  target coarse shape:", tuple(y_target_small.shape))

In [ ]:
# Fast two-level seed search: coarse solver for screening + fine solver for top-k
solver_coarse = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=1.0),
        method="euler",
        device=device,
        use_adjoint=False,
    ),
)

solver_fine = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=0.08),
        method="rk4",
        device=device,
        use_adjoint=False,
    ),
)


def set_model_speed(fast: bool):
    """Toggle fast/accurate Poisson inner iterations."""
    model.poisson_iterations = 8 if fast else 32


# Helper: update model params -> predict n(t, x)
def update_model_and_predict(model_parameters, timepoints=target_timepoints, solver_obj=solver_fine):
    # model_parameters = [B, Dn, k_s]
    B_val, Dn_val, ks_val = model_parameters

    model.B.data = torch.tensor(float(B_val), dtype=torch.float32, device=device)
    model.Dn.data = torch.tensor(float(Dn_val), dtype=torch.float32, device=device)
    model.k_s.data = torch.tensor(float(ks_val), dtype=torch.float32, device=device)

    with torch.inference_mode():
        _, traj = solver_obj.solve(timepoints=timepoints, u_initial=model.get_initial_state())
        pred_n = torch.clamp(extract_trajectory_component(traj, 0), 0.0, 1.0)
    return pred_n


def compute_sse_fast(
    params,
    target_tensor,
    timepoints,
    solver_obj,
    early_stop_threshold=None,
    downsample_factor=1,
    fast_poisson=False,
):
    old_poisson = int(model.poisson_iterations)
    set_model_speed(fast_poisson)

    try:
        with torch.inference_mode():
            # Early reject on first follow-up (not initial condition).
            if early_stop_threshold is not None and len(timepoints) > 1 and target_tensor.shape[0] > 1:
                pred_2 = update_model_and_predict(params, timepoints=timepoints[:2], solver_obj=solver_obj)[1:2]
                tgt_2 = target_tensor[1:2]
                if downsample_factor > 1:
                    pred_2 = downsample_field(pred_2, downsample_factor)
                    tgt_2 = downsample_field(tgt_2, downsample_factor)
                sse_2 = torch.sum((pred_2 - tgt_2) ** 2).item()
                if sse_2 > early_stop_threshold:
                    return float("inf")

            pred = update_model_and_predict(params, timepoints=timepoints, solver_obj=solver_obj)
            tgt = target_tensor
            if downsample_factor > 1:
                pred = downsample_field(pred, downsample_factor)
                tgt = downsample_field(tgt, downsample_factor)
            return torch.sum((pred - tgt) ** 2).item()
    finally:
        model.poisson_iterations = old_poisson


def compute_sse(params, target_tensor, timepoints):
    return compute_sse_fast(
        params,
        target_tensor,
        timepoints,
        solver_obj=solver_fine,
        early_stop_threshold=None,
        downsample_factor=1,
        fast_poisson=False,
    )


def estimate_radius(n_field, threshold=0.01):
    volume_voxels = (n_field > threshold).sum().float()
    volume_voxels = torch.clamp(volume_voxels, min=1.0)
    return (3.0 * volume_voxels / (4.0 * torch.pi)) ** (1.0 / 3.0)


def estimate_initial_params(initial_n, final_n, dt_days):
    ttc_0 = torch.clamp(initial_n.sum(), min=1e-8)
    ttc_1 = torch.clamp(final_n.sum(), min=1e-8)

    B_est = torch.log(ttc_1 / ttc_0) / max(float(dt_days), 1e-6)
    B_est = torch.clamp(B_est, 0.005, 0.1)

    radius_0 = estimate_radius(initial_n)
    radius_1 = estimate_radius(final_n)
    v_front = (radius_1 - radius_0) / max(float(dt_days), 1e-6)

    Dn_est = (v_front**2) / torch.clamp(4.0 * B_est, min=1e-8)
    Dn_est = torch.clamp(Dn_est, 5e-4, 0.02)

    return float(B_est.item()), float(Dn_est.item()), float(v_front.item())


def pre_calibration_grid_search_fast(
    y_target_full,
    y_target_small_local,
    timepoints_local,
    b_bounds,
    dn_bounds,
    k_bounds,
    n_coarse_visits=2,
    top_k_fine=5,
):
    import time

    if len(timepoints_local) >= 2:
        dt_days = (timepoints_local[1] - timepoints_local[0]).total_seconds() / 86400.0
        B_est, Dn_est, v_front = estimate_initial_params(y_target_full[0], y_target_full[1], dt_days)
    else:
        B_est, Dn_est, v_front = 0.02, 0.003, 0.0

    print(f"Analytic init: B={B_est:.4f}, Dn={Dn_est:.4f}, v_front={v_front:.4f} vox/day")

    b_mult = [0.6, 0.8, 1.0, 1.2, 1.5]
    dn_mult = [0.5, 0.8, 1.0, 1.3, 2.0]

    B_grid = sorted({float(np.clip(B_est * m, b_bounds[0], b_bounds[1])) for m in b_mult})
    Dn_grid = sorted({float(np.clip(Dn_est * m, dn_bounds[0], dn_bounds[1])) for m in dn_mult})
    k_grid = [float(np.clip(k, k_bounds[0], k_bounds[1])) for k in [0.03, 0.08, 0.15, 0.25]]

    total_combos = len(B_grid) * len(Dn_grid) * len(k_grid)
    print(f"Coarse grid: {total_combos} combinations")

    n_eval_visits = min(max(2, n_coarse_visits), len(timepoints_local))
    coarse_timepoints = timepoints_local[:n_eval_visits]
    coarse_target_small = y_target_small_local[:n_eval_visits]

    # Heuristic threshold for fast rejection at first follow-up.
    early_stop = float(coarse_target_small[1].numel()) * 0.5 if n_eval_visits > 1 else None

    t0 = time.time()
    candidates = []
    skipped = 0

    for B in B_grid:
        for Dn in Dn_grid:
            for ks in k_grid:
                params = torch.tensor([B, Dn, ks], dtype=torch.float64)
                sse = compute_sse_fast(
                    params,
                    coarse_target_small,
                    coarse_timepoints,
                    solver_obj=solver_coarse,
                    early_stop_threshold=early_stop,
                    downsample_factor=COARSE_FACTOR,
                    fast_poisson=True,
                )
                if np.isfinite(sse):
                    candidates.append((sse, params))
                else:
                    skipped += 1

    candidates.sort(key=lambda x: x[0])
    if not candidates:
        raise RuntimeError("All coarse candidates were rejected. Relax early-stop threshold.")

    t1 = time.time()
    print(f"Level 1 done: valid={len(candidates)}, skipped={skipped}, elapsed={t1 - t0:.1f}s")

    fine_candidates = candidates[: min(top_k_fine, len(candidates))]
    best_sse = float("inf")
    best_params = None

    for rank, (_, params) in enumerate(fine_candidates, start=1):
        sse_full = compute_sse_fast(
            params,
            y_target_full,
            timepoints_local,
            solver_obj=solver_fine,
            early_stop_threshold=None,
            downsample_factor=1,
            fast_poisson=False,
        )
        print(
            f"  Fine [{rank}/{len(fine_candidates)}]: "
            f"B={params[0]:.4f} Dn={params[1]:.4f} k_s={params[2]:.4f} -> SSE={sse_full:.4e}"
        )
        if sse_full < best_sse:
            best_sse = sse_full
            best_params = params

    t2 = time.time()
    print(f"Level 2 elapsed: {t2 - t1:.1f}s | Total elapsed: {t2 - t0:.1f}s")
    print(
        f"Grid best: B={best_params[0]:.4f}, Dn={best_params[1]:.4f}, "
        f"k_s={best_params[2]:.4f}, SSE={best_sse:.4e}"
    )
    return best_params, best_sse


# Baseline error before calibration (default params)
params_pre_cal = torch.tensor([model.B.item(), model.Dn.item(), model.k_s.item()], dtype=torch.float64)
baseline_sse = compute_sse(params_pre_cal, y_target, target_timepoints)
print(f"Baseline SSE (default params): {baseline_sse:.4e}")

# Better initial guess: analytic estimate + fast two-level coarse-to-fine search
search_bounds = (
    (0.005, 0.080),
    (0.0005, 0.020),
    (0.020, 0.300),
)

seed_parameters, seed_sse = pre_calibration_grid_search_fast(
    y_target_full=y_target,
    y_target_small_local=y_target_small,
    timepoints_local=target_timepoints,
    b_bounds=search_bounds[0],
    dn_bounds=search_bounds[1],
    k_bounds=search_bounds[2],
    n_coarse_visits=2,
    top_k_fine=5,
)
print(f"Seed SSE before LM: {seed_sse:.4e}")

In [ ]:
# LM calibration setup
initial_parameters = seed_parameters.clone().to(dtype=torch.float64)
print("LM init params (from pre-search):", initial_parameters)

# bounds for [B, Dn, k_s]
bounds = torch.tensor(
    [
        [0.005, 0.080],   # B
        [0.0005, 0.020],  # Dn
        [0.020, 0.300],   # k_s
    ],
    dtype=torch.float64,
)

lm_options = LMoptions(
    jac_delta=1e-4,
    jac_update_interval=2,  # recompute Jacobian every 2 accepted steps (faster)
    lambda_init=1.0,
    lambda_upscale_factor=5.0,
    lambda_downscale_factor=1.5,
)

optim = LMoptimizer(
    model=update_model_and_predict,
    bounds=bounds,
    initial_guess=initial_parameters,
    y_data=y_target,
    options=lm_options,
)

n_optim_steps = 6
for i in range(n_optim_steps):
    optim.step()
    print(f"step {i+1:02d}: best_error={optim.error[-1]:.4e}, params={optim.parameters[-1].tolist()}")

best_parameters = optim.parameters[-1]
print("Best parameters:", best_parameters)

In [ ]:
# Apply best parameters and compare against targets
model.B.data = torch.tensor(float(best_parameters[0]), dtype=torch.float32, device=device)
model.Dn.data = torch.tensor(float(best_parameters[1]), dtype=torch.float32, device=device)
model.k_s.data = torch.tensor(float(best_parameters[2]), dtype=torch.float32, device=device)

pred_cal = update_model_and_predict(best_parameters)
cal_sse = torch.sum((pred_cal - y_target) ** 2).item()

print(f"Baseline SSE (default): {baseline_sse:.4e}")
print(f"Seed SSE (grid best) : {seed_sse:.4e}")
print(f"Calibrated SSE (LM)  : {cal_sse:.4e}")

# Loss trace
plt.figure(figsize=(6, 3.5))
plt.plot(optim.error, marker="o")
plt.title("LM calibration loss (best SSE)")
plt.xlabel("iteration")
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Compare mean tumor density at calibration visits
pred_mean = pred_cal.mean(dim=(1, 2, 3)).detach().cpu().numpy()
tgt_mean = y_target.mean(dim=(1, 2, 3)).detach().cpu().numpy()
visit_days = np.array([(t - target_timepoints[0]).days for t in target_timepoints], dtype=float)

plt.figure(figsize=(6, 3.5))
plt.plot(visit_days, tgt_mean, "o-", label="target mean n")
plt.plot(visit_days, pred_mean, "s--", label="predicted mean n (calibrated)")
plt.title("Calibration fit at visit times")
plt.xlabel("days from first visit")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

## Quick vs full calibration presets

This section adds two ready-to-run calibration modes:

- `quick`: fewer LM iterations and fewer visits (fast debug)
- `full`: more iterations and more visits (better fit, slower)

Run one preset and then compare **day-50 tumor slice before vs after calibration**.

In [ ]:
calibration_presets = {
    "quick": {
        "n_visits": min(4, len(patient_data.visits)),
        "n_steps": 8,
        "solver_step_days": 0.05,
        "late_visit_weight": 2.5,
        "roi_weight": 4.0,
        "roi_threshold": 0.02,
    },
    "full": {
        "n_visits": min(6, len(patient_data.visits)),
        "n_steps": 20,
        "solver_step_days": 0.03,
        "late_visit_weight": 3.5,
        "roi_weight": 6.0,
        "roi_threshold": 0.02,
    },
}


def run_hemo_calibration(preset_name="quick"):
    cfg = calibration_presets[preset_name]

    # Save current params to compare before/after
    params_before = torch.tensor(
        [
            model.B.item(),
            model.Dn.item(),
            model.Ds.item(),
            model.k_s.item(),
            model.s_crit.item(),
        ],
        dtype=torch.float64,
    )

    target_t = [v.time for v in patient_data.visits[: cfg["n_visits"]]]
    y_tgt = torch.stack(
        [
            torch.from_numpy(measured_cellularity_maps[i].array).float().to(device)
            for i in range(cfg["n_visits"])
        ],
        dim=0,
    )

    # Later visits typically encode the strongest treatment response signal.
    visit_weights = torch.linspace(1.0, cfg["late_visit_weight"], steps=len(target_t), device=device)
    roi_union = (torch.max(y_tgt, dim=0).values > cfg["roi_threshold"]).float()
    voxel_weights = torch.ones_like(roi_union) + (cfg["roi_weight"] - 1.0) * roi_union

    # Temporarily adjust solver step for calibration workload
    old_step = solver.solver_options.step_size
    solver.solver_options.step_size = timedelta(days=cfg["solver_step_days"])

    def _predict(params, timepoints=target_t):
        B_val, Dn_val, Ds_val, ks_val, s_crit_val = params
        model.B.data = torch.tensor(float(B_val), dtype=torch.float32, device=device)
        model.Dn.data = torch.tensor(float(Dn_val), dtype=torch.float32, device=device)
        model.Ds.data = torch.tensor(float(Ds_val), dtype=torch.float32, device=device)
        model.k_s.data = torch.tensor(float(ks_val), dtype=torch.float32, device=device)
        model.s_crit.data = torch.tensor(float(s_crit_val), dtype=torch.float32, device=device)
        _, traj = solver.solve(timepoints=timepoints, u_initial=model.get_initial_state())
        return torch.clamp(extract_trajectory_component(traj, 0), 0.0, 1.0)

    def _compute_sse(params):
        with torch.no_grad():
            residual = _predict(params) - y_tgt
            weighted = (residual**2) * visit_weights[:, None, None, None] * voxel_weights[None, ...]
            return torch.mean(weighted).item()

    try:
        bounds = torch.tensor(
            [
                [0.003, 0.080],   # B
                [0.0002, 0.020],  # Dn
                [0.002, 0.060],   # Ds
                [0.010, 0.350],   # k_s
                [0.150, 0.650],   # s_crit
            ],
            dtype=torch.float64,
        )

        if len(target_t) >= 2:
            dt_days = max((target_t[1] - target_t[0]).total_seconds() / 86400.0, 1e-6)
            B_est, Dn_est, _ = estimate_initial_params(y_tgt[0], y_tgt[1], dt_days)
        else:
            B_est, Dn_est = 0.02, 0.003

        B_grid = sorted(
            {
                float(np.clip(B_est * m, bounds[0, 0].item(), bounds[0, 1].item()))
                for m in [0.6, 0.85, 1.0, 1.2, 1.5]
            }
        )
        Dn_grid = sorted(
            {
                float(np.clip(Dn_est * m, bounds[1, 0].item(), bounds[1, 1].item()))
                for m in [0.5, 0.8, 1.0, 1.25, 1.6]
            }
        )
        k_grid = [0.02, 0.04, 0.08, 0.15, 0.25]

        ds_seed = float(np.clip(model.Ds.item(), bounds[2, 0].item(), bounds[2, 1].item()))
        s_crit_seed = float(np.clip(model.s_crit.item(), bounds[4, 0].item(), bounds[4, 1].item()))

        seed = None
        seed_sse = float("inf")
        for B in B_grid:
            for Dn in Dn_grid:
                for ks in k_grid:
                    p = torch.tensor([B, Dn, ds_seed, ks, s_crit_seed], dtype=torch.float64)
                    sse = _compute_sse(p)
                    if sse < seed_sse:
                        seed_sse = sse
                        seed = p

        init_guess = seed.clone()

        optim_local = LMoptimizer(
            model=_predict,
            bounds=bounds,
            initial_guess=init_guess,
            y_data=y_tgt,
            options=LMoptions(jac_delta=5e-5, jac_update_interval=1, lambda_init=0.5),
        )

        for _ in range(cfg["n_steps"]):
            optim_local.step()

        best = optim_local.parameters[-1]
        model.B.data = torch.tensor(float(best[0]), dtype=torch.float32, device=device)
        model.Dn.data = torch.tensor(float(best[1]), dtype=torch.float32, device=device)
        model.Ds.data = torch.tensor(float(best[2]), dtype=torch.float32, device=device)
        model.k_s.data = torch.tensor(float(best[3]), dtype=torch.float32, device=device)
        model.s_crit.data = torch.tensor(float(best[4]), dtype=torch.float32, device=device)

        baseline = _compute_sse(params_before)
        calibrated = _compute_sse(best)

        return {
            "preset": preset_name,
            "config": cfg,
            "params_before": params_before,
            "params_seed": init_guess,
            "params_after": best,
            "baseline_sse": baseline,
            "seed_sse": seed_sse,
            "calibrated_sse": calibrated,
            "optim": optim_local,
        }
    finally:
        solver.solver_options.step_size = old_step


# Choose preset: "quick" or "full"
cal_result = run_hemo_calibration("quick")
print("Preset:", cal_result["preset"])
print("Params before:", cal_result["params_before"])
print("Params seed  :", cal_result["params_seed"])
print("Params after :", cal_result["params_after"])
print(f"Weighted SSE before: {cal_result['baseline_sse']:.4e}")
print(f"Weighted SSE seed  : {cal_result['seed_sse']:.4e}")
print(f"Weighted SSE after : {cal_result['calibrated_sse']:.4e}")

plt.figure(figsize=(6, 3.2))
plt.plot(cal_result["optim"].error, marker="o")
plt.title(f"LM loss ({cal_result['preset']})")
plt.xlabel("iteration")
plt.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
# Day-50 tumor slice before vs after calibration

def run_50d_n_series(params, step_days=0.05):
    old_step = solver.solver_options.step_size
    solver.solver_options.step_size = timedelta(days=step_days)

    try:
        model.B.data = torch.tensor(float(params[0]), dtype=torch.float32, device=device)
        model.Dn.data = torch.tensor(float(params[1]), dtype=torch.float32, device=device)
        model.k_s.data = torch.tensor(float(params[2]), dtype=torch.float32, device=device)

        t0 = patient_data.visits[0].time
        tp = [t0 + timedelta(days=d) for d in range(0, 51)]
        tdays, traj = solver.solve(timepoints=tp, u_initial=model.get_initial_state())
        tumor_series = torch.clamp(extract_trajectory_component(traj, 0), 0.0, 1.0)
        return tdays.detach().cpu().numpy(), tumor_series
    finally:
        solver.solver_options.step_size = old_step


_, n_before = run_50d_n_series(cal_result["params_before"], step_days=0.05)
_, n_after = run_50d_n_series(cal_result["params_after"], step_days=0.05)

z = n_before.shape[1] // 2
idx50 = 50

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

im0 = axes[0].imshow(n_before[idx50, z].detach().cpu().numpy(), cmap="magma")
axes[0].set_title("n day50 before calibration")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(n_after[idx50, z].detach().cpu().numpy(), cmap="magma")
axes[1].set_title("n day50 after calibration")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

delta = (n_after[idx50, z] - n_before[idx50, z]).detach().cpu().numpy()
im2 = axes[2].imshow(delta, cmap="coolwarm")
axes[2].set_title("delta (after - before)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()

## TumorTwin-style TCC/map check

This reproduces the same quick check pattern from `HGG_Demo` / `TNBC_Demo`:

- total tumor cell count curve,
- cellularity maps at selected days.

In [ ]:
# Equivalent of:
# fig, ax = plt.subplots(1, 1, figsize=(5, 2))
# plot_predicted_TCC(predicted_cellularity_maps, timepoints, ax=ax)
# ... and map snapshots at [0, 50, 100, 150, 200] days.

# Use full-run tumor trajectory, projected to physical range
predicted_cellularity_maps = [torch.clamp(u_t, 0.0, 1.0) for u_t in n_full]
timepoints = full_timepoints

fig, ax = plt.subplots(1, 1, figsize=(5, 2))
plot_predicted_TCC(predicted_cellularity_maps, timepoints, ax=ax)
plt.tight_layout()

requested_days = [0, 50, 100, 150, 200]
time_days = np.array([days_since_first(t, timepoints[0]) for t in timepoints])

available_days = [d for d in requested_days if d <= time_days.max()]
if len(available_days) < len(requested_days):
    print(
        f"Requested days {requested_days}, but max simulated day is {time_days.max():.1f}. "
        f"Using {available_days} instead."
    )

fig, axes = plt.subplots(1, len(available_days), figsize=(max(5, len(available_days) * 2.2), 2))
if len(available_days) == 1:
    axes = [axes]

for i, t in enumerate(available_days):
    # robust index lookup (nearest simulated output)
    t_idx = int(np.argmin(np.abs(time_days - t)))
    plot_cellularity_map(
        predicted_cellularity_maps[t_idx], patient_data, time=int(t), ax=axes[i]
    )

plt.tight_layout()

## Short full-run (HGG-style plots)

This section runs the full framework pipeline on a **short horizon** and reproduces the core HGG-style visual checks:

- patient timeline + imaging summary,
- predicted TCC (with measured visit TCC overlay),
- predicted cellularity snapshots,
- predicted vs measured maps at visit times within the short horizon.

In [ ]:
# 1) Framework context plots (same style as HGG demo)
plot_patient_timeline(patient_data)
plot_imaging_summary(patient_data)

In [ ]:
# 2) Short full-run (small horizon, but full pipeline)
SHORT_HORIZON_DAYS = 60
SHORT_OUTPUT_STEP_DAYS = 0.5

t0 = patient_data.visits[0].time
t_end_short = t0 + timedelta(days=SHORT_HORIZON_DAYS)

timepoints_short = []
cur_t = t0
while cur_t <= t_end_short:
    timepoints_short.append(cur_t)
    cur_t += timedelta(days=SHORT_OUTPUT_STEP_DAYS)
if timepoints_short[-1] != t_end_short:
    timepoints_short.append(t_end_short)

_, traj_short = solver.solve(timepoints=timepoints_short, u_initial=model.get_initial_state())
pred_short_n = torch.clamp(extract_trajectory_component(traj_short, 0), 0.0, 1.0)  # tumor channel only

print("Short full-run complete")
print("  horizon (days):", SHORT_HORIZON_DAYS)
print("  outputs:", len(timepoints_short))
print("  pred shape:", tuple(pred_short_n.shape))
print("  finite:", bool(torch.isfinite(pred_short_n).all()))

In [ ]:
# 3) TCC plot (predicted) + measured visit TCC overlay
predicted_cellularity_maps_short = [u_t for u_t in pred_short_n]

fig, ax = plt.subplots(1, 1, figsize=(5, 2))
plot_predicted_TCC(predicted_cellularity_maps_short, timepoints_short, ax=ax)

# measured TCC at visits inside the short horizon
measured_maps_short = []
measured_days_short = []
for visit in patient_data.visits:
    d = days_since_first(visit.time, t0)
    if d <= SHORT_HORIZON_DAYS:
        cell_map = ADC_to_cellularity(
            visit.adc_image,
            visit.roi_enhance_image,
            visit.roi_nonenhance_image,
        )
        measured_maps_short.append(torch.from_numpy(cell_map.array).float())
        measured_days_short.append(d)

if measured_maps_short:
    measured_tcc = [
        compute_total_cell_count(m, carrying_capacity=carrying_capacity).item()
        for m in measured_maps_short
    ]
    ax.scatter(measured_days_short, measured_tcc, c="tab:red", s=20, label="measured visits")
    ax.legend(loc="best")

plt.tight_layout()

In [ ]:
# 4) Predicted maps at selected times (HGG-like quick panel)
requested_days = [0, 15, 30, 45, 60]
time_days_short = np.array([days_since_first(t, timepoints_short[0]) for t in timepoints_short])

fig, axes = plt.subplots(1, len(requested_days), figsize=(5, 2))
for i, d in enumerate(requested_days):
    t_idx = int(np.argmin(np.abs(time_days_short - d)))
    plot_cellularity_map(predicted_cellularity_maps_short[t_idx], patient_data, time=d, ax=axes[i])

plt.tight_layout()

In [ ]:
# 5) Predicted vs measured maps at visit times within short horizon
visit_idxs_short = [
    i for i, v in enumerate(patient_data.visits)
    if days_since_first(v.time, t0) <= SHORT_HORIZON_DAYS
]

if len(visit_idxs_short) == 0:
    print("No visit falls inside short horizon.")
else:
    fig, axs = plt.subplots(2, len(visit_idxs_short), figsize=(max(5, 2.2 * len(visit_idxs_short)), 4))
    if len(visit_idxs_short) == 1:
        axs = np.array(axs).reshape(2, 1)

    for col, vi in enumerate(visit_idxs_short):
        vd = days_since_first(patient_data.visits[vi].time, t0)
        pred_idx = int(np.argmin(np.abs(time_days_short - vd)))

        measured_map = ADC_to_cellularity(
            patient_data.visits[vi].adc_image,
            patient_data.visits[vi].roi_enhance_image,
            patient_data.visits[vi].roi_nonenhance_image,
        )

        plot_cellularity_map(predicted_cellularity_maps_short[pred_idx], patient_data, time=int(vd), ax=axs[0, col])
        plot_cellularity_map(torch.from_numpy(measured_map.array).float(), patient_data, time=int(vd), ax=axs[1, col])

    axs[0, 0].set_ylabel("pred", rotation=90)
    axs[1, 0].set_ylabel("meas", rotation=90)
    plt.tight_layout()